# GRPO 3 回目: 場面の時刻を当てる（4B、LoRA、Colab A100）

用途を「問いが指す場面の時刻を答える」に絞った最初の学習。設計は `docs/scene_search.md`。

- 学習データ: 日本の CC BY 動画から作った場面 `data/ja/train_time.jsonl`（1 場面 = 1 問、答えは時刻区間）
- 報酬: 答えの時刻が区間 ± 許容幅に入れば 1.0、外れた秒数で線形に減る + 形式 0.1。出力は根拠 1〜2 行 + 「答え: 分:秒」（96 トークン）
- 設定: LoRA r=16、32 フレーム、G=8、4 問/ステップ、lr 1e-5、β 0.04、動的サンプリング（上限 12 問）
- 1 ステップ 2〜3 分の見込み。400 ステップは 1 セッションに収まらないので、**切れたら同じセルを再実行**（`latest.txt` から続く）

前提: Drive の `MyDrive/movie_llm/` に `ja_frames_f32_s640.zip`（日本の映像のフレーム）と `frames_f32_s640.zip`（ベンチマークのフレーム）がある。

評価の順: 学習前の dev → 学習 → 学習後の dev → ベンチマークの時刻（722 問）→ ベンチマークの 4 択（壊れていないか）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

## 2. Drive とフレーム

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
OUT = f'{DRIVE}/grpo3_time'
import os
!mkdir -p /content/ja_frames /content/frames {OUT}
!unzip -q -o {DRIVE}/ja_frames_f32_s640.zip -d /content/ja_frames
!unzip -q -o {DRIVE}/frames_f32_s640.zip -d /content/frames
for frames, videos in (('/content/ja_frames', '/content/ja_videos'), ('/content/frames', '/content/videos')):
    os.makedirs(videos, exist_ok=True)
    for vid in os.listdir(frames):
        open(f'{videos}/{vid}.mp4', 'a').close()
    print(frames, len(os.listdir(frames)), '本')

## 3. 学習前の dev（貪欲法）

In [ ]:
%cd /content/movie_llm
!python scripts/pt_difficulty.py --items data/ja/dev_time.jsonl --out {OUT}/dev_before.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --greedy --video-dir /content/ja_videos --frames-dir /content/ja_frames

## 4. GRPO（まず 20 ステップで時間を測る → 400 に伸ばす。切れたら同じセルを再実行）

In [ ]:
%cd /content/movie_llm
!python scripts/train_grpo.py --items data/ja/train_time.jsonl --out {OUT} \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --video-dir /content/ja_videos --frames-dir /content/ja_frames \
    --group-size 8 --prompts-per-step 4 --micro-batch 2 --max-new-tokens 96 \
    --lr 1e-5 --beta 0.04 --max-draws 12 --max-steps 20 --save-every 20

In [ ]:
import json, pandas as pd
rows = {}
for l in open(f'{OUT}/train_log.jsonl'):
    d = json.loads(l); rows[d['step']] = d
df = pd.DataFrame(sorted(rows.values(), key=lambda d: d['step']))
print(df[['step','reward','acc','parsable','kl','loss','len','draws','zero_adv_groups','elapsed']].tail(20).to_string(index=False))
print('1 ステップあたり', round(df.elapsed.diff().clip(lower=0).median()), '秒')
w = df.groupby((df.step - 1) // 50).agg(reward=('reward','mean'), acc=('acc','mean'), kl=('kl','mean'), len=('len','mean'), draws=('draws','mean'), skip=('zero_adv_groups','mean'))
print(w.round(3).to_string())

## 5. 学習後の dev

In [ ]:
%cd /content/movie_llm
latest = open(f'{OUT}/latest.txt').read().strip()
print('adapter:', latest)
!python scripts/pt_difficulty.py --items data/ja/dev_time.jsonl --out {OUT}/dev_after.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --greedy --video-dir /content/ja_videos --frames-dir /content/ja_frames --adapter {latest}
!python scripts/pt_difficulty.py --items data/ja/dev_time.jsonl --out {OUT}/dev_before.jsonl --summarize
!python scripts/pt_difficulty.py --items data/ja/dev_time.jsonl --out {OUT}/dev_after.jsonl --summarize

## 6. ベンチマーク: 場面の時刻（722 問、約 25 分）と 4 択（220 問、約 5 分）

In [ ]:
%cd /content/movie_llm
latest = open(f'{OUT}/latest.txt').read().strip()
!python scripts/jvqa_predict.py --name grpo3_time-4b_f32 --task time --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --max-new-tokens 96 --adapter {latest} \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results
!python scripts/jvqa_eval_time.py --predictions {DRIVE}/results/grpo3_time-4b_f32/predictions.jsonl --name grpo3_time-4b_f32 --out-root {DRIVE}/results
!python scripts/jvqa_predict.py --name grpo3_mc-4b_f32 --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --adapter {latest} \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results --type multi_choice

終わったら、手順 4 の表、手順 5 の 2 つの集計、手順 6 の時刻の表を手元の私に貼る。持ち帰るもの:
- `MyDrive/movie_llm/grpo3_time/` の `latest.txt` が指す LoRA フォルダと `train_log.jsonl` → 手元の `results/grpo3_time/`
- `MyDrive/movie_llm/results/grpo3_time-4b_f32/predictions.jsonl` と `grpo3_mc-4b_f32/predictions.jsonl` → 手元の `results/` の同名フォルダ